# Lab 02 · Credit Scoring with Machine Learning
**AI in Finance · Module 2**

<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A bank must decide who gets credit. In this lab you build the model behind that decision, then check whether it is **accurate**, **profitable**, **explainable** and **fair**.

**By the end of this lab you will be able to:**
1. Prepare real credit data and spot data-quality problems.
2. Train and compare a logistic regression and a gradient-boosted model.
3. Evaluate credit models with AUC, Gini and KS, and choose a decision threshold using business costs.
4. Convert a model into a points-based **credit scorecard**.
5. Explain individual decisions with **SHAP** (reason codes).
6. Audit the model for **fairness** with **Fairlearn**.

**Dataset:** *Default of Credit Card Clients* (Taiwan, 2005), 30,000 customers, UCI Machine Learning Repository. Target: did the customer default on next month's payment?

**How to run:** `File → Save a copy in Drive`, then `Runtime → Run all`. A CPU runtime is enough. Time needed: about 2 hours.

> Cells marked **✏️ Your turn** are exercises. Answer the **🤔 Think** questions in a text cell under each one.

## 0 · Setup
Installs the few libraries Colab does not already include. Takes about 30 seconds.

In [ ]:
!pip install -q ucimlrepo shap fairlearn xgboost

In [ ]:
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
SEED = 42
np.random.seed(SEED)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## 1 · Load the data
The loader tries the UCI repository first, then OpenML as a backup. If both fail (for example, no internet), upload the CSV your instructor provides using the 📁 Files panel and set `LOCAL_CSV`.

In [ ]:
COLS = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
        "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
        "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
        "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]
LOCAL_CSV = None   # e.g. "credit_default.csv" (23 feature columns + a 'DEFAULT' column)

def load_credit_data():
    if LOCAL_CSV:
        return pd.read_csv(LOCAL_CSV)
    try:
        from ucimlrepo import fetch_ucirepo
        d = fetch_ucirepo(id=350)
        X, y = d.data.features.copy(), d.data.targets.iloc[:, 0]
        print("Loaded from UCI")
    except Exception as e:
        print("UCI failed:", e, "\nTrying OpenML...")
        from sklearn.datasets import fetch_openml
        d = fetch_openml(data_id=42477, as_frame=True, parser="auto")
        X, y = d.data.copy(), d.target
        print("Loaded from OpenML")
    X = X.iloc[:, -23:]          # keep the 23 features in their documented order
    X.columns = COLS
    df = X.apply(pd.to_numeric)
    df["DEFAULT"] = pd.to_numeric(y).astype(int).values
    return df

df = load_credit_data()
print(df.shape)
df.head()

**Data dictionary**

| Column | Meaning |
|---|---|
| LIMIT_BAL | Credit limit (NT dollars) |
| SEX | 1 = male, 2 = female *(protected attribute, used only for the fairness audit)* |
| EDUCATION | 1 = graduate school, 2 = university, 3 = high school, 4 = other |
| MARRIAGE | 1 = married, 2 = single, 3 = other |
| AGE | Age in years |
| PAY_0, PAY_2 … PAY_6 | Repayment status for Sep → Apr: −2/−1/0 = paid on time or no use; 1–9 = months of delay |
| BILL_AMT1–6 | Bill statement amount, Sep → Apr |
| PAY_AMT1–6 | Amount paid, Sep → Apr |
| DEFAULT | 1 = defaulted next month |

## 2 · Explore and clean

In [ ]:
rate = df["DEFAULT"].mean()
print(f"Customers: {len(df):,}   Default rate: {rate:.1%}")
print("\nUnexpected codes check:")
print("EDUCATION values:", sorted(df["EDUCATION"].unique()))
print("MARRIAGE values: ", sorted(df["MARRIAGE"].unique()))

🤔 **Think:** The data dictionary lists EDUCATION codes 1–4 and MARRIAGE codes 1–3, but the data contains other values. What could they mean, and what are the risks of (a) deleting those rows, (b) merging them into "other"?

We merge them into "other" below and record that decision.

In [ ]:
df["EDUCATION"] = df["EDUCATION"].where(df["EDUCATION"].isin([1, 2, 3]), 4)
df["MARRIAGE"]  = df["MARRIAGE"].where(df["MARRIAGE"].isin([1, 2]), 3)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.groupby("PAY_0")["DEFAULT"].mean().plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set(title="Default rate by most recent repayment status", xlabel="PAY_0 (months late)", ylabel="Default rate")
pd.qcut(df["LIMIT_BAL"], 5, duplicates="drop").pipe(lambda b: df.groupby(b, observed=True)["DEFAULT"].mean()).plot.bar(ax=axes[1], color="#DD8452")
axes[1].set(title="Default rate by credit-limit band", xlabel="LIMIT_BAL quintile", ylabel="")
plt.tight_layout(); plt.show()

## 3 · Feature engineering
Credit analysts rarely use raw columns alone. We add three ratios that bankers actually look at.

In [ ]:
bills = df[[f"BILL_AMT{i}" for i in range(1, 7)]]
pays  = df[[f"PAY_AMT{i}" for i in range(1, 7)]]
status = df[["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]]

df["UTILISATION"]   = (df["BILL_AMT1"] / df["LIMIT_BAL"]).clip(-1, 3)        # how much of the limit is used
df["PAY_RATIO"]     = (pays.sum(axis=1) / bills.clip(lower=0).sum(axis=1).replace(0, np.nan)).fillna(1).clip(0, 3)
df["MONTHS_LATE"]   = (status > 0).sum(axis=1)                               # months with any delay

PROTECTED = "SEX"
FEATURES = [c for c in df.columns if c not in ("DEFAULT", PROTECTED)]
print(len(FEATURES), "features:", FEATURES)

🤔 **Think:** We left `SEX` out of the model's inputs. Does that guarantee the model treats men and women equally? (You will test this in Section 8.)

## 4 · Train / test split
We keep 25% of customers aside and never touch them until the final evaluation. `stratify` keeps the default rate the same in both parts.

In [ ]:
X = df[FEATURES]; y = df["DEFAULT"]; s = df[PROTECTED]
X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y, s, test_size=0.25, stratify=y, random_state=SEED)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}  Default rate train/test: {y_train.mean():.1%} / {y_test.mean():.1%}")

## 5 · Two models
- **Logistic regression**: the industry standard for scorecards, easy to explain to regulators.
- **Gradient boosting (XGBoost)**: usually more accurate, harder to explain.

In [ ]:
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
logit.fit(X_train, y_train)

try:
    from xgboost import XGBClassifier
    gbm = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                        colsample_bytree=0.8, eval_metric="auc", random_state=SEED,
                        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum())
    GBM_NAME = "XGBoost"
except ImportError:  # fallback so the lab still runs without XGBoost
    from sklearn.ensemble import HistGradientBoostingClassifier
    gbm = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300,
                                         class_weight="balanced", random_state=SEED)
    GBM_NAME = "HistGradientBoosting"
gbm.fit(X_train, y_train)

p_logit = logit.predict_proba(X_test)[:, 1]
p_gbm   = gbm.predict_proba(X_test)[:, 1]
print("Models trained:", "LogisticRegression,", GBM_NAME)

## 6 · Evaluate like a credit-risk analyst
Accuracy is misleading here: predicting "nobody defaults" is already about 78% accurate. Credit teams use:
- **AUC** (0.5 = random, 1 = perfect) and **Gini = 2·AUC − 1**
- **KS statistic**: the largest gap between the score distributions of good and bad customers
- **PR-AUC**: precision–recall area, useful when defaults are rare

In [ ]:
def ks_stat(y_true, p):
    fpr, tpr, _ = roc_curve(y_true, p)
    return np.max(tpr - fpr)

def credit_metrics(name, y_true, p):
    auc = roc_auc_score(y_true, p)
    return {"Model": name, "AUC": auc, "Gini": 2 * auc - 1, "KS": ks_stat(y_true, p),
            "PR-AUC": average_precision_score(y_true, p)}

results = pd.DataFrame([credit_metrics("Logistic regression", y_test, p_logit),
                        credit_metrics(GBM_NAME, y_test, p_gbm)]).set_index("Model").round(3)
display(results)

for name, p in [("Logistic regression", p_logit), (GBM_NAME, p_gbm)]:
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, p):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC curves"); plt.legend(); plt.show()

In [ ]:
# Is the difference real or luck? 5-fold cross-validation on the training data
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
for name, m in [("Logistic regression", logit), (GBM_NAME, gbm)]:
    sc = cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name:22s} CV AUC = {sc.mean():.3f} ± {sc.std():.3f}")

🤔 **Think:** Is the more accurate model worth the loss of transparency? Write two sentences a bank's model-risk committee would accept.

## 7 · From probability to decision: choose the threshold with business costs
The model outputs a probability of default; the bank still has to pick a cut-off. Mistakes cost different amounts:
- **Missed defaulter** (false negative): the bank loses the money lent.
- **Rejected good customer** (false positive): the bank loses the interest it would have earned.

In [ ]:
COST_FN = 5.0   # cost of approving a customer who defaults (relative units)
COST_FP = 1.0   # cost of rejecting a customer who would have repaid

def expected_cost(y_true, p, t):
    pred = (p >= t).astype(int)                      # 1 = predicted default = reject
    fn = ((pred == 0) & (y_true == 1)).sum()
    fp = ((pred == 1) & (y_true == 0)).sum()
    return (COST_FN * fn + COST_FP * fp) / len(y_true)

thresholds = np.linspace(0.05, 0.95, 91)
costs = [expected_cost(y_test.values, p_gbm, t) for t in thresholds]
best_t = thresholds[int(np.argmin(costs))]

plt.plot(thresholds, costs); plt.axvline(best_t, ls="--", c="r", label=f"best threshold = {best_t:.2f}")
plt.axvline(0.5, ls=":", c="grey", label="default 0.5")
plt.xlabel("Threshold (reject if P(default) ≥ t)"); plt.ylabel("Expected cost per customer"); plt.legend(); plt.show()

pred = (p_gbm >= best_t).astype(int)
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["Repaid", "Default"]).plot(cmap="Blues")
plt.title(f"{GBM_NAME} at threshold {best_t:.2f}"); plt.show()
print(f"Approval rate: {(pred == 0).mean():.1%}")

✏️ **Your turn:** Change `COST_FN` to 2 and then to 10, and re-run the cell. How does the best threshold move, and what happens to the approval rate? Explain in business terms.

## 8 · Build a points-based scorecard
Banks communicate risk as a **score** (like 300–850), not a probability. The industry formula uses *points to double the odds* (PDO):

Score = Offset + Factor × ln(odds of repaying), with Factor = PDO / ln 2

Here a score of **600** means odds of 50 : 1 of repaying, and every **20 points** doubles those odds.

In [ ]:
BASE_SCORE, BASE_ODDS, PDO = 600, 50, 20
factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)

def to_score(p_default):
    p = np.clip(p_default, 1e-6, 1 - 1e-6)
    return np.round(offset + factor * np.log((1 - p) / p))

# class_weight="balanced" inflates probabilities, so refit a plain logistic model for calibrated scores
logit_cal = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_train, y_train)
scores = to_score(logit_cal.predict_proba(X_test)[:, 1])

bands = pd.cut(scores, bins=[-np.inf, 520, 560, 600, 640, np.inf], labels=["<520", "520–559", "560–599", "600–639", "640+"])
table = pd.DataFrame({"band": bands, "default": y_test.values}).groupby("band", observed=True)["default"].agg(["count", "mean"])
table.columns = ["Customers", "Default rate"]
display(table.style.format({"Default rate": "{:.1%}"}))

🤔 **Think:** A good scorecard shows default rates falling steadily as scores rise. Does yours? If one band breaks the pattern, what could cause it?

## 9 · Explain the decisions with SHAP
Regulators and customers can ask *"why was I rejected?"*. SHAP shows how much each feature pushed a prediction up or down.

In [ ]:
import shap
sample = X_test.sample(500, random_state=SEED)
try:
    explainer = shap.TreeExplainer(gbm)           # fast, exact for tree models
    sv = explainer(sample)
except Exception as e:                             # fallback if SHAP and XGBoost versions clash
    print("TreeExplainer unavailable (", type(e).__name__, ") - using the slower model-agnostic explainer")
    background = shap.sample(X_train, 100, random_state=SEED)
    explainer = shap.Explainer(lambda d: gbm.predict_proba(d)[:, 1], background)
    sv = explainer(sample)
shap.plots.beeswarm(sv, max_display=12)

In [ ]:
# Reason codes for one rejected applicant
p_sample = gbm.predict_proba(sample)[:, 1]
i = int(np.argmax(p_sample >= best_t))            # first rejected applicant in the sample
print(f"Applicant P(default) = {p_sample[i]:.2f} → REJECTED (threshold {best_t:.2f})")
shap.plots.waterfall(sv[i], max_display=10)

contrib = pd.Series(sv[i].values, index=sample.columns).sort_values(ascending=False)
print("Top 3 reasons for rejection:")
for feat, val in contrib.head(3).items():
    print(f"  • {feat} = {sample.iloc[i][feat]}  (raised the risk estimate by {val:+.3f})")

✏️ **Your turn:** Write the three reasons above as a short, plain-language letter to the customer (no jargon such as "log-odds"). Which reason could the customer actually act on?

## 10 · Fairness audit with Fairlearn
We removed `SEX` from the inputs, but other features may act as **proxies**. We compare outcomes for men (1) and women (2):
- **Selection rate**: share predicted to default, i.e. rejected
- **True positive rate**: share of real defaulters caught
- **False positive rate**: share of good customers wrongly rejected

In [ ]:
from fairlearn.metrics import (MetricFrame, selection_rate, true_positive_rate, false_positive_rate,
                               demographic_parity_difference, equalized_odds_difference)

mf = MetricFrame(metrics={"Rejection rate": selection_rate, "TPR": true_positive_rate, "FPR": false_positive_rate},
                 y_true=y_test, y_pred=pred, sensitive_features=s_test.map({1: "Male", 2: "Female"}))
display(mf.by_group.round(3))
print(f"Demographic parity difference: {demographic_parity_difference(y_test, pred, sensitive_features=s_test):.3f}")
print(f"Equalized odds difference:     {equalized_odds_difference(y_test, pred, sensitive_features=s_test):.3f}")
mf.by_group.plot.bar(rot=0, title="Outcomes by group"); plt.show()

🤔 **Think:** A common rule of thumb flags differences above about 0.05–0.10. Is your model fair? Which fairness definition matters more for lending: equal rejection rates, or equal error rates? There is no single correct answer, so justify yours.

In [ ]:
# Optional: post-processing mitigation (group-specific thresholds) and its cost to accuracy
from fairlearn.postprocessing import ThresholdOptimizer
to = ThresholdOptimizer(estimator=gbm, constraints="equalized_odds", prefit=True, predict_method="predict_proba")
to.fit(X_train, y_train, sensitive_features=s_train)
pred_fair = to.predict(X_test, sensitive_features=s_test, random_state=SEED)
print(f"Equalized odds difference: before {equalized_odds_difference(y_test, pred, sensitive_features=s_test):.3f}"
      f"  →  after {equalized_odds_difference(y_test, pred_fair, sensitive_features=s_test):.3f}")
print(f"Expected cost per customer: before {expected_cost(y_test.values, p_gbm, best_t):.3f}"
      f"  →  after {(COST_FN*((pred_fair==0)&(y_test.values==1)).sum() + COST_FP*((pred_fair==1)&(y_test.values==0)).sum())/len(y_test):.3f}")

🤔 **Think:** Using different thresholds for men and women can improve fairness metrics, but it means treating applicants differently by sex, which many regulators forbid. How would you resolve this tension?

## 11 · Model card
Every production model needs a one-page summary. This cell builds it from your results.

In [ ]:
card = f'''
MODEL CARD · Credit default model
Model:            {GBM_NAME} (challenger) vs logistic regression (benchmark)
Data:             UCI Default of Credit Card Clients, {len(df):,} customers, default rate {df["DEFAULT"].mean():.1%}
Test AUC / Gini:  {results.loc[GBM_NAME, "AUC"]:.3f} / {results.loc[GBM_NAME, "Gini"]:.3f}
Decision rule:    reject if P(default) >= {best_t:.2f}  (cost ratio FN:FP = {COST_FN:.0f}:{COST_FP:.0f})
Approval rate:    {(pred == 0).mean():.1%}
Excluded inputs:  SEX (kept only for fairness testing)
Limitations:      2005 data from Taiwan; may not reflect today's or UAE customers.
                  Fairness checked for sex only; age and marital status not audited.
'''
print(card)

In [ ]:
# Optional: save the model to Google Drive
SAVE = False
if SAVE:
    import joblib
    from google.colab import drive
    drive.mount("/content/drive")
    joblib.dump(gbm, "/content/drive/MyDrive/credit_model.joblib")
    print("Saved to Drive")

## 12 · Exercises to submit

| # | Task | Marks |
|---|---|---|
| 1 | Answer all 🤔 **Think** questions in text cells (2–4 sentences each). | 4 |
| 2 | **Threshold:** complete the ✏️ cost-ratio exercise in Section 7 and recommend one threshold to the bank's credit committee. | 2 |
| 3 | **Improve the model:** add one new engineered feature of your own (for example, the trend in bill amounts). Report whether test AUC improves and whether the improvement is larger than the cross-validation spread. | 3 |
| 4 | **Reason codes:** write the customer letter from Section 9. | 2 |
| 5 | **Fairness:** repeat the audit using `AGE` groups (under 30, 30–45, over 45). Report what you find and one possible cause. | 3 |
| 6 | **Model card:** fill in two more limitations you discovered during the lab. | 1 |
|  | **Total** | **15** |

**Submit:** `File → Download → Download .ipynb`, rename it `RegNo_Lab02.ipynb`, and upload it to the LMS. Make sure all outputs are visible (run the whole notebook before downloading).

**AI-use rule:** you may ask an AI tool to explain an error message, but your interpretations and answers must be your own. Declare any AI use in a final text cell.